In [1]:
!pip install hnswlib

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp314-cp314-win_amd64.whl size=187374 sha256=153fff90488077989a1fafae660dabc0723604268599c39f4ae55a97ae0d948f
  Stored in directory: c:\users\andhi\appdata\local\pip\cache\wheels\54\48\f6\bd9aec7940a15fab891fd004077f7ad37141c5653273fb7514
Successfully built hnswlib



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

In [3]:
# Build Dummy Dataset
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # search for 5 nearest neighbors

In [4]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performance
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Time:", end - start, "s")

=== Exact NN ===
Indices: [[903 785 359 840 146]]
Distances: [[0.02626913 0.02711362 0.03000126 0.0326854  0.03854402]]
Time: 0.05153799057006836 s


In [5]:
# Initiate Index
p = hnswlib.Index(space='l2', dim=dim)  

# Define Max Elements
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Add Data
p.add_items(data)

# Set searching parameter
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Time:", end - start, "s")


=== HNSW ===
Indices: [[903 785 359 840 146]]
Distances: [[0.00069007 0.00073515 0.00090008 0.00106834 0.00148564]]
Time: 0.0001068115234375 s
